In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load validation data

df_val = pd.read_csv("fct_val_cleaned.csv", low_memory=False)
print(f"Raw shape: {df_val.shape}")
print(f"\nOutcome distribution:")
print(df_val['outcome_from_filename'].value_counts())

In [ ]:
df_val.columns = (
    df_val.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)
print(f"Columns: {df_val.columns.tolist()}")

In [ ]:
df_val = df_val[
    ~df_val['no'].astype(str).str.lower().isin(['no', 'nan'])
]
df_val = df_val[
    ~df_val['step_name_1'].astype(str).str.lower().str.contains(
        'step name', na=False
    )
]
df_val.reset_index(drop=True, inplace=True)
print(f"After cleaning: {df_val.shape}")

df_val['unit_id'] = df_val['source_file'].astype(str).str.strip()
print(f"Unique units: {df_val['unit_id'].nunique()}")

In [ ]:
df_val['result'] = (
    df_val['result'].astype(str).str.strip().str.upper()
)
df_val['outcome_from_filename'] = (
    df_val['outcome_from_filename']
    .astype(str).str.strip().str.upper()
)
df_val['result_binary'] = df_val['result'].map(
    {'PASS': 1, 'FAIL': 0, 'ABORT': 0}
)
df_val['unit_pass'] = df_val['outcome_from_filename'].map(
    {'PASS': 1, 'FAIL': 0, 'ABORT': 0}
)
df_val['step_time']   = pd.to_numeric(
    df_val['step_time'], errors='coerce'
).fillna(0.0)
df_val['step_name_1'] = (
    df_val['step_name_1'].astype(str).str.strip()
)
df_val['step_name_2'] = (
    df_val['step_name_2'].astype(str).str.strip()
)
df_val['test_id'] = (
    df_val['step_name_1'] + " / " + df_val['step_name_2']
)

print(f"Result values:   {df_val['result'].unique()}")
print(f"Unit outcomes:   {df_val['outcome_from_filename'].unique()}")
print(f"Unique test_ids: {df_val['test_id'].nunique()}")

In [ ]:
unit_summary_val = df_val.groupby('unit_id').agg(
    overall_outcome=('unit_pass', 'first'),
    total_test_time=('step_time', 'sum'),
    num_steps_executed=('test_id', 'count'),
    num_steps_failed=('result_binary', lambda x: (x == 0).sum())
).reset_index()


all_failing_val = unit_summary_val[
    unit_summary_val['overall_outcome'] == 0
]['unit_id'].tolist()

genuine_fail_val = []
abort_only_val   = []

for unit_id in all_failing_val:
    unit_steps    = df_val[df_val['unit_id'] == unit_id]
    has_step_fail = (unit_steps['result'] == 'FAIL').any()
    if has_step_fail:
        genuine_fail_val.append(unit_id)
    else:
        abort_only_val.append(unit_id)

failing_units_val = genuine_fail_val
N_fail_val        = len(failing_units_val)

n_total = len(unit_summary_val)
n_pass  = unit_summary_val['overall_outcome'].sum()
n_fail  = (unit_summary_val['overall_outcome'] == 0).sum()

print(f"\n{'='*55}")
print("VALIDATION DATASET SUMMARY (Jan-Feb FCT)")
print(f"{'='*55}")
print(f"Total units:          {n_total}")
print(f"Passing units:        {n_pass}")
print(f"Failing units:        {n_fail}")
print(f"Genuine FAIL units:   {N_fail_val}")
print(f"ABORT-only excluded:  {len(abort_only_val)}")
print(f"Defect rate:          "
      f"{N_fail_val/n_total*100:.1f}%")
print(f"Unique test steps:    {df_val['test_id'].nunique()}")

In [ ]:
outcome_matrix_val = df_val.pivot_table(
    index='unit_id',
    columns='test_id',
    values='result_binary',
    aggfunc='first'
)

cost_vector_val = df_val.groupby(
    'test_id'
)['step_time'].mean().round(4)

C_full_cost_val = cost_vector_val.sum()

print(f"Outcome matrix:       {outcome_matrix_val.shape}")
print(f"Full sequence cost:   {C_full_cost_val:.2f} s")

In [ ]:
# Load reduced test plan 
cred_df   = pd.read_csv('fct_rq2_cred.csv')
C_red_val = cred_df['test_id'].tolist()
print(f"C_red steps: {len(C_red_val)}")
print(f"First 3: {C_red_val[:3]}")

In [ ]:
# Match training C_red steps to validation column names
cred_df = pd.read_csv("fct_rq2_cred.csv")
C_red   = cred_df['test_id'].tolist()

val_steps = set(outcome_matrix_val.columns.tolist())
C_red_set = set(C_red)

in_both       = C_red_set & val_steps
only_in_train = C_red_set - val_steps

print(f"Cred steps (training):        {len(C_red_set)}")
print(f"Cred steps in validation:     {len(in_both)}")
print(f"Cred steps NOT in validation: {len(only_in_train)}")

# Fix case mismatches between training and validation step names
case_fixes = {}
for missing in only_in_train:
    matches = [
        v for v in val_steps
        if missing.lower() == v.lower()
    ]
    if matches:
        case_fixes[missing] = matches[0]

C_red_val = []
for s in C_red:
    if s in val_steps:
        C_red_val.append(s)
    elif s in case_fixes:
        C_red_val.append(case_fixes[s])

print(f"\nFinal Cred steps for validation: {len(C_red_val)}")

In [ ]:
C_red_cost_val = sum(
    float(cost_vector_val[s]) 
    if s in cost_vector_val.index else 0.0
    for s in C_red_val
)
saving_static_val = (
    C_full_cost_val - C_red_cost_val
) / C_full_cost_val * 100

# Build validation detection matrix
fail_matrix_val = outcome_matrix_val.loc[
    outcome_matrix_val.index.isin(failing_units_val)
].copy()
detection_val = (fail_matrix_val == 0).astype(float).fillna(0)

missing_in_val = [
    s for s in C_red_val 
    if s not in detection_val.columns
]
print(f"C_red steps missing in validation: {len(missing_in_val)}")

cred_cols_val   = [
    c for c in C_red_val 
    if c in detection_val.columns
]
cred_detect_val = detection_val[cred_cols_val].sum(axis=1)
escaped_static  = (cred_detect_val == 0).sum()
detected_static = (cred_detect_val > 0).sum()

escape_risk_static = escaped_static / N_fail_val

print(f"\n{'='*55}")
print("STATIC CRED — VALIDATION (Jan-Feb FCT)")
print(f"{'='*55}")
print(f"C_red steps (training):   {len(C_red_val)}")
print(f"C_red steps in val data:  {len(cred_cols_val)}")
print(f"Failing units:            {N_fail_val}")
print(f"Detected by Cred:         {detected_static}")
print(f"Escaped:                  {escaped_static}")
print(f"Escape risk (ε):          {escape_risk_static:.6f}")
print(f"Cost saving:              {saving_static_val:.2f}%")

In [ ]:
# MAB hyperparameters and validation stream setup
PASS_RATE_WINDOW    = 50
PASS_RATE_THRESHOLD = 0.93
ESCAPE_PENALTY      = 10

unit_outcome_map_val = unit_summary_val.set_index(
    'unit_id'
)['overall_outcome'].to_dict()

seen_val        = set()
unit_stream_val = []
for u in df_val['unit_id'].tolist():
    if u not in seen_val:
        unit_stream_val.append(u)
        seen_val.add(u)

unit_pass_seq_val = np.array([
    unit_outcome_map_val.get(u, 1)
    for u in unit_stream_val
])

rolling_pass_rate_val = np.zeros(len(unit_stream_val))
for i in range(len(unit_stream_val)):
    window_start = max(0, i - PASS_RATE_WINDOW + 1)
    window_vals  = unit_pass_seq_val[window_start:i+1]
    rolling_pass_rate_val[i] = window_vals.mean()

exec_index_val = np.arange(len(unit_stream_val))

print(f"Units in stream:      {len(unit_stream_val)}")
print(f"Rolling pass rate:")
print(f"  Mean:               {rolling_pass_rate_val.mean():.4f}")
print(f"  Min:                {rolling_pass_rate_val.min():.4f}")
print(f"  % stable:           "
      f"{(rolling_pass_rate_val >= PASS_RATE_THRESHOLD).mean()*100:.1f}%")
print(f"  % unstable:         "
      f"{(rolling_pass_rate_val < PASS_RATE_THRESHOLD).mean()*100:.1f}%")

In [ ]:
# MAB helper functions for validation
def unit_is_defective_val(unit_id):
    return unit_id in failing_units_val

def cred_detects_val(unit_id):
    if unit_id not in outcome_matrix_val.index:
        return True
    row = outcome_matrix_val.loc[unit_id]
    for step in C_red_val:
        if step in row.index and row[step] == 0:
            return True
    return False

def run_mab_val(boost_factor, seed=42):
    """
    Run Thompson Sampling MAB on validation data
    with given instability boost factor.
    """
    alpha = [5.0, 1.0]
    beta  = [1.0, 1.0]

    choices  = []
    escaped  = 0
    detected = 0
    costs    = []

    np.random.seed(seed)

    for i, unit_id in enumerate(unit_stream_val):
        pass_rate = rolling_pass_rate_val[i]
        stable    = pass_rate >= PASS_RATE_THRESHOLD

        theta_full = np.random.beta(alpha[0], beta[0])
        theta_red  = np.random.beta(alpha[1], beta[1])

        if not stable:
            theta_full += (
                PASS_RATE_THRESHOLD - pass_rate
            ) * boost_factor

        chosen = 1 if theta_red > theta_full else 0
        choices.append(chosen)

        reward, det, esc = get_reward_val(chosen, unit_id)
        costs.append(
            C_full_cost_val if chosen == 0 else C_red_cost_val
        )

        if det:
            detected += 1
        if esc:
            escaped += 1

        reward_clipped = max(0.0, min(1.0, reward))
        if reward_clipped >= 0.5:
            alpha[chosen] += 1
        else:
            beta[chosen] += 1

    pct_cred   = round(choices.count(1) / len(choices) * 100, 1)
    mean_cost  = np.mean(costs)
    saving_pct = round(
        (C_full_cost_val - mean_cost) / C_full_cost_val * 100, 2
    )
    esc_risk   = round(escaped / N_fail_val, 6)

    return {
        'pct_cred':   pct_cred,
        'saving_pct': saving_pct,
        'escaped':    escaped,
        'detected':   detected,
        'escape_risk': esc_risk
    }

print("Validation helper functions defined")

In [ ]:
# Reward function for validation
def get_reward_val(chosen_arm, unit_id):
    is_defective = unit_is_defective_val(unit_id)
    detected     = False
    escaped      = False

    if chosen_arm == 0:
        reward = 0.0
        if is_defective:
            detected = True
    else:
        if not is_defective:
            reward = 1.0
        elif cred_detects_val(unit_id):
            reward   = 0.5
            detected = True
        else:
            reward  = -ESCAPE_PENALTY
            escaped = True

    return reward, detected, escaped

print("Reward function defined")

In [ ]:
print(f"\n{'='*75}")
print("FINAL RESULTS TABLE — FCT VALIDATION")
print(f"{'='*75}")

# Baseline
print(f"Jan-Feb  Baseline (Cfull)  ---  0.0%  0.00%  0 escapes")

# Static C_red
static_escaped = sum(
    1 for u in unit_stream_val
    if unit_is_defective_val(u) and not cred_detects_val(u)
)
static_saving = (
    (C_full_cost_val - C_red_cost_val) / C_full_cost_val * 100
)
print(f"Jan-Feb  Static Cred (RQ1)  ---  "
      f"100.0%  {static_saving:.2f}%  "
      f"{static_escaped} escapes")

# Thompson Sampling results per beta value
for boost in [10, 50]:
    r = run_mab_val(boost_factor=boost)
    print(f"Jan-Feb  Thompson Sampling  "
          f"β={boost}  "
          f"{r['pct_cred']}%  "
          f"{r['saving_pct']}%  "
          f"{r['escaped']} escapes")

In [ ]:
# Build validation detection matrix
fail_matrix_val = outcome_matrix_val.loc[
    outcome_matrix_val.index.isin(failing_units_val)
].copy()
detection_val = (fail_matrix_val == 0).astype(float).fillna(0)

# Match C_red steps to validation columns
cred_cols_val = [
    c for c in C_red_val 
    if c in detection_val.columns
]

cred_detect_val = detection_val[cred_cols_val].sum(axis=1)
escaped_static  = (cred_detect_val == 0).sum()
detected_static = (cred_detect_val > 0).sum()

escape_risk_static = escaped_static / N_fail_val

print(f"Failing units:          {N_fail_val}")
print(f"Detected by Cred:       {detected_static}")
print(f"Escaped:                {escaped_static}")
print(f"Escape risk:            {escape_risk_static:.6f}")

In [ ]:
# Find which steps catch the escaped units
escaped_unit_ids = detection_val[
    detection_val[cred_cols_val].sum(axis=1) == 0
].index.tolist()

non_cred_steps = [
    s for s in detection_val.columns
    if s not in set(C_red_val)
]

step_coverage = {}
for step in non_cred_steps:
    if step not in detection_val.columns:
        continue
    n_detected = detection_val.loc[
        escaped_unit_ids, step
    ].sum()
    if n_detected > 0:
        step_coverage[step] = int(n_detected)

step_coverage_df = pd.DataFrame([
    {'test_id': k, 'escaped_units_detected': v}
    for k, v in step_coverage.items()
]).sort_values('escaped_units_detected', ascending=False)

print(f"{'='*55}")
print("CONCEPT DRIFT ANALYSIS")
print(f"{'='*55}")
print(f"Escaped units:        {len(escaped_unit_ids)}")
print(f"\nSteps that detect escaped units (not in Cred):")
print(step_coverage_df.to_string(index=False))
print(f"\nKey finding: {step_coverage_df.iloc[0]['test_id']}")
print(f"Detects {step_coverage_df.iloc[0]['escaped_units_detected']} "
      f"of {len(escaped_unit_ids)} escaped units")
print(f"This step had ZERO detections in training period")

In [ ]:
result_b10 = run_mab_val(boost_factor=10)

print(f"\n{'='*55}")
print("MAB AGENT (boost=10) — VALIDATION (Jan-Feb FCT)")
print(f"{'='*55}")
print(f"Chose Cred:           {result_b10['pct_cred']:.1f}%")
print(f"Cost saving:          {result_b10['saving_pct']:.2f}%")
print(f"Escaped:              {result_b10['escaped']}")
print(f"Escape risk (ε):      {result_b10['escape_risk']:.6f}")


In [ ]:
boost_factors = [10, 20, 30, 50, 100]
sensitivity   = []

# Sensitivity analysis across instability boost values
for boost in boost_factors:
    r = run_mab_val(boost_factor=boost)
    sensitivity.append({
        'Boost factor':  boost,
        'Cred (%)':      r['pct_cred'],
        'Saving (%)':    r['saving_pct'],
        'Escaped':       r['escaped'],
        'Escape risk ε': r['escape_risk']
    })

sens_df = pd.DataFrame(sensitivity)
sens_df.to_csv("fct_val_sensitivity.csv", index=False)

print(f"\n{'='*65}")
print("SENSITIVITY ANALYSIS — BOOST FACTOR (Jan-Feb FCT)")
print(f"{'='*65}")
print(sens_df.to_string(index=False))

In [ ]:
result_b50 = run_mab_val(boost_factor=50)

print(f"\n{'='*55}")
print("MAB AGENT (boost=50) — VALIDATION (Jan-Feb FCT)")
print(f"{'='*55}")
print(f"Chose Cred:           {result_b50['pct_cred']:.1f}%")
print(f"Cost saving:          {result_b50['saving_pct']:.2f}%")
print(f"Escaped:              {result_b50['escaped']}")
print(f"Escape risk (ε):      {result_b50['escape_risk']:.6f}")

In [ ]:
# Compile final results table — training and validation
rows = [
    # Training period
    {
        'Period':     'Jul-Dec (train)',
        'Algorithm':  'Baseline (Cfull)',
        'Boost':      '---',
        'Cred (%)':   0.0,
        'Saving (%)': 0.00,
        'Saving (s)': 0.00,
        'Escaped':    0,
        'ε':          0.000000
    },
    {
        'Period':     'Jul-Dec (train)',
        'Algorithm':  'Static Cred (RQ1)',
        'Boost':      '---',
        'Cred (%)':   100.0,
        'Saving (%)': 18.78,
        'Saving (s)': 29.60,
        'Escaped':    0,
        'ε':          0.000000
    },
    {
        'Period':     'Jul-Dec (train)',
        'Algorithm':  'Epsilon-Greedy',
        'Boost':      '10',
        'Cred (%)':   89.6,
        'Saving (%)': 16.83,
        'Saving (s)': round(16.83/100 * C_full_cost_val, 2),
        'Escaped':    0,
        'ε':          0.000000
    },
    {
        'Period':     'Jul-Dec (train)',
        'Algorithm':  'UCB',
        'Boost':      '10',
        'Cred (%)':   92.8,
        'Saving (%)': 17.42,
        'Saving (s)': round(17.42/100 * C_full_cost_val, 2),
        'Escaped':    0,
        'ε':          0.000000
    },
    {
        'Period':     'Jul-Dec (train)',
        'Algorithm':  'Thompson Sampling',
        'Boost':      '10',
        'Cred (%)':   93.8,
        'Saving (%)': 17.61,
        'Saving (s)': round(17.61/100 * C_full_cost_val, 2),
        'Escaped':    0,
        'ε':          0.000000
    },
    # Validation period
    {
        'Period':     'Jan-Feb (val)',
        'Algorithm':  'Baseline (Cfull)',
        'Boost':      '---',
        'Cred (%)':   0.0,
        'Saving (%)': 0.00,
        'Saving (s)': 0.00,
        'Escaped':    0,
        'ε':          0.000000
    },
    {
        'Period':     'Jan-Feb (val)',
        'Algorithm':  'Static Cred (RQ1)',
        'Boost':      '---',
        'Cred (%)':   100.0,
        'Saving (%)': round(saving_static_val, 2),
        'Saving (s)': round(
            saving_static_val/100 * C_full_cost_val, 2
        ),
        'Escaped':    int(escaped_static),
        'ε':          round(escape_risk_static, 6)
    },
    {
        'Period':     'Jan-Feb (val)',
        'Algorithm':  'Thompson Sampling',
        'Boost':      '10',
        'Cred (%)':   result_b10['pct_cred'],
        'Saving (%)': result_b10['saving_pct'],
        'Saving (s)': round(
            result_b10['saving_pct']/100 * C_full_cost_val, 2
        ),
        'Escaped':    result_b10['escaped'],
        'ε':          result_b10['escape_risk']
    },
    {
        'Period':     'Jan-Feb (val)',
        'Algorithm':  'Thompson Sampling',
        'Boost':      '50',
        'Cred (%)':   result_b50['pct_cred'],
        'Saving (%)': result_b50['saving_pct'],
        'Saving (s)': round(
            result_b50['saving_pct']/100 * C_full_cost_val, 2
        ),
        'Escaped':    result_b50['escaped'],
        'ε':          result_b50['escape_risk']
    },
]

final_df = pd.DataFrame(rows)
final_df.to_csv("fct_val_final_results.csv", index=False)

print(f"\n{'='*75}")
print("FINAL RESULTS TABLE — FCT TRAINING + VALIDATION")
print(f"{'='*75}")
print(final_df.to_string(index=False))

In [ ]:
# ── Figure 1 — Rolling pass rate validation ─────────────────
fig1, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(
    exec_index_val, rolling_pass_rate_val,
    color='black', linewidth=1.5,
    label=f'Rolling pass rate (w={PASS_RATE_WINDOW})'
)
ax1.axhline(
    y=PASS_RATE_THRESHOLD, color='red',
    linestyle='--', linewidth=1.5,
    label=f'Stability threshold ({PASS_RATE_THRESHOLD})'
)
ax1.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax1.set_ylabel(
    f'Rolling pass rate (w={PASS_RATE_WINDOW})',
    fontsize=14, fontweight='bold'
)
ax1.set_xlim(0, 6000)
ax1.set_ylim(0, 1)
ax1.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax1.xaxis.set_major_locator(plt.MultipleLocator(600))
ax1.legend(fontsize=10)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='both', labelsize=11)
plt.tight_layout()
plt.savefig('fct_val_rolling_passrate.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_val_rolling_passrate.png")

# ── Figure 2 — Arm selection validation (boost=10) ──────────
window = 300
choices_arr_val = np.array(result_b10['choices'])
rolling_cred_val = pd.Series(choices_arr_val).rolling(
    window=window, min_periods=1
).mean().values

fig2, ax2 = plt.subplots(figsize=(7, 5))
ax2.plot(
    exec_index_val, rolling_cred_val,
    color='black', linewidth=2,
    label=f'Fraction selecting Cred (w={window}, boost=10)'
)
ax2.axhline(
    y=0.5, color='red', linestyle='--',
    linewidth=1.5, label='50% threshold'
)
ax2.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax2.set_ylabel(
    'Fraction selecting Cred',
    fontsize=14, fontweight='bold'
)
ax2.set_xlim(0, 6000)
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax2.xaxis.set_major_locator(plt.MultipleLocator(600))
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='both', labelsize=11)
ax2.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fct_val_arm_selection.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_val_arm_selection.png")

In [ ]:
# Shared style function
def style_ax(ax):
    ax.xaxis.set_major_locator(plt.MaxNLocator(10))
    ax.yaxis.set_major_locator(plt.MaxNLocator(10))
    ax.tick_params(axis='both', labelsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)

exec_index_val = np.arange(len(unit_stream_val))
window         = 300

# Figure 1 — Rolling pass rate validation
fig1, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(
    exec_index_val, rolling_pass_rate_val,
    color='black', linewidth=1.5,
    label=f'Rolling pass rate (w={PASS_RATE_WINDOW})'
)
ax1.axhline(
    y=PASS_RATE_THRESHOLD, color='red',
    linestyle='--', linewidth=1.5,
    label=f'Stability threshold ({PASS_RATE_THRESHOLD})'
)
ax1.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax1.set_ylabel(
    f'Rolling pass rate (w={PASS_RATE_WINDOW})',
    fontsize=14, fontweight='bold'
)
ax1.set_ylim(0, 1)
ax1.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax1.xaxis.set_major_locator(plt.MultipleLocator(600))
ax1.legend(fontsize=10)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='both', labelsize=11)
plt.tight_layout()
plt.savefig('fct_val_rolling_passrate.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_val_rolling_passrate.png")

# Figure 2 — Arm selection validation (boost=10)
choices_arr_val  = np.array(result_b10['choices'])
rolling_cred_val = pd.Series(choices_arr_val).rolling(
    window=window, min_periods=1
).mean().values

fig2, ax2 = plt.subplots(figsize=(7, 5))
ax2.plot(
    exec_index_val, rolling_cred_val,
    color='black', linewidth=2,
    label=f'Fraction selecting Cred (w={window}, boost=10)'
)
ax2.axhline(
    y=0.5, color='red', linestyle='--',
    linewidth=1.5, label='50% threshold'
)
ax2.set_xlabel(
    'Execution index (FCT events in time order)',
    fontsize=14, fontweight='bold'
)
ax2.set_ylabel(
    'Fraction selecting Cred',
    fontsize=14, fontweight='bold'
)
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax2.xaxis.set_major_locator(plt.MultipleLocator(600))
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='both', labelsize=11)
ax2.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fct_val_arm_selection.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_val_arm_selection.png")

# Figure 3 — Sensitivity analysis bar chart
fig3, axes = plt.subplots(1, 2, figsize=(12, 5))

boost_vals  = sens_df['Boost factor'].tolist()
saving_vals = sens_df['Saving (%)'].tolist()
escape_vals = sens_df['Escape risk ε'].tolist()

axes[0].bar(
    [str(b) for b in boost_vals],
    saving_vals,
    color='steelblue', alpha=0.8,
    edgecolor='black', linewidth=0.5
)
axes[0].set_xlabel(
    'Instability sensitivity parameter (β)',
    fontsize=13, fontweight='bold'
)
axes[0].set_ylabel(
    'Cost Saving (%)',
    fontsize=13, fontweight='bold'
)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(labelsize=11)
for j, v in enumerate(saving_vals):
    axes[0].text(
        j, v + 0.05, f'{v:.2f}%',
        ha='center', fontsize=9, fontweight='bold'
    )

axes[1].bar(
    [str(b) for b in boost_vals],
    escape_vals,
    color='steelblue', alpha=0.8,
    edgecolor='black', linewidth=0.5
)
axes[1].set_xlabel(
    'Instability sensitivity parameter (β)',
    fontsize=13, fontweight='bold'
)
axes[1].set_ylabel(
    'Escape risk (ε)',
    fontsize=14, fontweight='bold'
)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(labelsize=11)

plt.tight_layout()
plt.savefig('fct_val_sensitivity.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_val_sensitivity.png")

In [ ]:
fig3, axes = plt.subplots(1, 2, figsize=(12, 5))

boost_vals  = sens_df['Boost factor'].tolist()
saving_vals = sens_df['Saving (%)'].tolist()
escape_vals = sens_df['Escape risk ε'].tolist()

# Left — cost saving
axes[0].bar(
    [str(b) for b in boost_vals],
    saving_vals,
    color='black', alpha=0.8,
    edgecolor='black', linewidth=0.5
)
axes[0].set_xlabel(
    'Instability sensitivity parameter (β)',
    fontsize=13, fontweight='bold'
)
axes[0].set_ylabel(
    'Cost Saving (%)',
    fontsize=13, fontweight='bold'
)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(labelsize=11)
for j, v in enumerate(saving_vals):
    axes[0].text(
        j, v + 0.05, f'{v:.2f}%',
        ha='center', fontsize=9, fontweight='bold'
    )

# Right — escape risk
axes[1].bar(
    [str(b) for b in boost_vals],
    escape_vals,
    color='black', alpha=0.8,
    edgecolor='black', linewidth=0.5
)
axes[1].set_xlabel(
    'Instability sensitivity parameter (β)',
    fontsize=13, fontweight='bold'
)
axes[1].set_ylabel(
    r'Escape risk $\hat{R}(\pi)$',
    fontsize=13, fontweight='bold'
)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(labelsize=11)
for j, v in enumerate(escape_vals):
    axes[1].text(
        j, v + 0.0001, f'{v:.4f}',
        ha='center', fontsize=9, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('fct_val_sensitivity.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fct_val_sensitivity.png")

In [ ]:
final_df.to_csv("fct_val_final_results.csv", index=False)
sens_df.to_csv("fct_val_sensitivity.csv", index=False)

print("All validation outputs saved:")
print("  fct_val_final_results.csv")
print("  fct_val_sensitivity.csv")
print("  fct_val_rolling_passrate.png")
print("  fct_val_arm_selection.png")
print("  fct_val_sensitivity.png")
